In [2]:
from google.colab import files

# This will open a file selection dialog. Please upload X_final.npy and y_final.npy.
uploaded = files.upload()

Saving X_final.npy to X_final.npy
Saving y_final.npy to y_final.npy


In [ ]:
# script.py

import pandas as pd
import numpy as np
import os

# --- 1. CONFIGURATION ---
DATA_DIR = 'raw_data'
WINDOW_SIZE = 180
def extract_heartbeats(record_id):
    ekg_path = os.path.join(DATA_DIR, f"{record_id}_ekg.csv")
    ann_path = os.path.join(DATA_DIR, f"{record_id}_annotations_1.csv")

    if not os.path.exists(ekg_path) or not os.path.exists(ann_path):
        return None, None

    df_signal = pd.read_csv(ekg_path)
    df_ann = pd.read_csv(ann_path)

    # --- COLUMN DETECTION ---
    if 'MLII' in df_signal.columns:
        signal = df_signal['MLII'].values
    else:
        # If MLII is missing, take the first column that isn't 'time'
        alt_col = [c for c in df_signal.columns if c.lower() != 'time'][0]
        print(f" Record {record_id}: 'MLII' not found. Using '{alt_col}' instead.   ")
        signal = df_signal[alt_col].values
    # ------------------------------------

    peak_indices = df_ann['index'].values
    labels = df_ann['annotation_symbol'].values

    beats, beat_labels = [], []
    half_win = WINDOW_SIZE // 2

    for i in range(len(peak_indices)):
        p = peak_indices[i]
        label = labels[i]

        if label in ['N', 'V'] and p > half_win and p < (len(signal) - half_win):
            segment = signal[p - half_win: p + half_win]

            denom = (np.max(segment) - np.min(segment))
            if denom == 0: continue
            norm_segment = (segment - np.min(segment)) / denom

            beats.append(norm_segment)
            beat_labels.append(0 if label == 'N' else 1)

    return np.array(beats), beat_labels

# --- 2. MAIN EXECUTION ---
# Auto-detect all IDs from /rawdata
record_ids = [f.split('_')[0] for f in os.listdir(DATA_DIR) if f.endswith('_ekg.csv')]
record_ids.sort()

all_x, all_y = [], []

print(f"Starting processing of {len(record_ids)} records...")

# Process each file with a progress bar
for i, rid in enumerate(record_ids):
    print(f"Processing [{i + 1}/{len(record_ids)}]: Record {rid}...", end='\r')

    bx, by = extract_heartbeats(rid)
    if bx is not None and len(bx) > 0:
        all_x.append(bx)
        all_y.extend(by)

print("\nProcessing complete! Balancing dataset now...")

# Combine into large arrays
X_raw = np.vstack(all_x)
y_raw = np.array(all_y)

# --- 3. BALANCE & ACCURACY ---
idx_n = np.where(y_raw == 0)[0]
idx_v = np.where(y_raw == 1)[0]

print(f"\nRaw Counts -> Normal (N): {len(idx_n)} | Abnormal (V): {len(idx_v)}")

# Downsampling to 50/50 split for maximum accuracy
np.random.seed(42)
idx_n_balanced = np.random.choice(idx_n, size=len(idx_v), replace=False)
balanced_indices = np.concatenate([idx_n_balanced, idx_v])
np.random.shuffle(balanced_indices)

X_final = X_raw[balanced_indices]
y_final = y_raw[balanced_indices]

# --- 4. SAVE CLEAN DATA ---
np.save('X_final.npy', X_final)
np.save('y_final.npy', y_final)

print(f"--- SUCCESS ---")
print(f"Final Balanced Dataset: {len(y_final)} total beats.")
print(f"Saved as X_final.npy and y_final.npy")

In [ ]:
# train_cnn.py


import setuptools
import sys

sys.modules['distutils'] = setuptools._distutils
import os

# 1. ENVIRONMENT FIXES
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import butter, lfilter  # NEW: For filtering

print("--- Starting Stabilized Deep Training Pipeline with DSP ---", flush=True)


# --- NEW: DSP FILTERING FUNCTION ---
def ecg_bandpass_filter(data, lowcut=0.5, highcut=45.0, fs=360, order=4):
    """
    Standard ECG Bandpass: 0.5Hz - 45Hz.
    Removes Baseline Wander and Powerline Interference.
    """
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return lfilter(b, a, data)


# --- 2. DATA LOADING & FILTERING ---
PILOT_MODE = False
try:
    X = np.load('X_final.npy')
    y = np.load('y_final.npy')

    if PILOT_MODE:
        X, y = X[:1000], y[:1000]
        print("!!! PILOT MODE ACTIVE !!!")

    print(f"Applying Bandpass Filter to {len(X)} samples...", flush=True)
    # Apply filter to each 180-sample segment
    X_filtered = np.array([ecg_bandpass_filter(sample) for sample in X])

    # Reshape for Conv1D: (Samples, Time Steps, Features)
    X = X_filtered.reshape(X_filtered.shape[0], 180, 1)
    print(f"Filtering complete.", flush=True)
except FileNotFoundError:
    print("ERROR: Data files not found.")
    sys.exit()

# --- 3. TRAIN/TEST SPLIT ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 4. STABILIZED 6-LAYER ARCHITECTURE ---
model = models.Sequential([
    layers.Input(shape=(180, 1)),
    layers.Conv1D(32, kernel_size=5, activation='relu', padding='same'),
    layers.MaxPooling1D(pool_size=4),
    layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'),
    layers.GlobalAveragePooling1D(),
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

opt = tf.keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# --- 5. TRAINING ---
EPOCHS = 30
history = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=32, validation_data=(X_test, y_test), verbose=1)

# --- 6. EVALUATION ---
y_probs = model.predict(X_test)
y_pred = (y_probs > 0.5).astype("int32")

# Metrics
acc, prec, rec, f1 = accuracy_score(y_test, y_pred), precision_score(y_test, y_pred), \
    recall_score(y_test, y_pred), f1_score(y_test, y_pred)

print("\n" + "=" * 35)
print(f" Accuracy:  {acc:.4%}")
print(f" Recall:    {rec:.4%}")
print("=" * 35)


# --- 7. PLOTTING RESULTS (Stability Check) ---
acc_plot = history.history['accuracy']
val_acc_plot = history.history['val_accuracy']
loss_plot = history.history['loss']
val_loss_plot = history.history['val_loss']
epochs_range = range(1, len(acc_plot) + 1)

plt.figure(figsize=(14, 6))

# Accuracy Plot
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc_plot, label='Training Accuracy', linewidth=2)
plt.plot(epochs_range, val_acc_plot, label='Validation Accuracy', linewidth=2)
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)

# Loss Plot
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss_plot, label='Training Loss', linewidth=2)
plt.plot(epochs_range, val_loss_plot, label='Validation Loss', linewidth=2)
plt.title('Model Loss (Binary Crossentropy)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('cnn_training_results.png')  # Automatically save for your slides
print("\nSuccess: Plots saved as 'cnn_training_results.png'")
plt.show()

# Save the resulting model for hardware weight extraction
model.save('ecg_model_stabilized.h5')
print(f"Success: Model weights saved as 'ecg_model_stabilized.h5'")

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'V-Beat'],
            yticklabels=['Normal', 'V-Beat'])
plt.title('Final Confusion Matrix')
plt.show()


# Weight extraction
print("\n--- Starting Hardware Parameter Extraction ---")

# Define fixed-point scaling factor (Q4.12 format: 1 sign bit, 3 integer bits, 12 fractional bits)
SCALE_FACTOR = 4096


# Convert floats to 16-bit Two's Complement Hexadecimal strings
def float_to_hex16(value, scale):
    int_val = int(round(value * scale))
    int_val = max(-32768, min(32767, int_val))  # Clip values to safe 16-bit boundaries

    if int_val < 0:
        int_val = (1 << 16) + int_val  # Convert to two's complement for negative numbers

    return f"{int_val:04X}"

# dedicated directory to store the hardware ROM initialization files
output_dir = 'hardware_roms'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Loop through each layer in your exact model sequence
for layer in model.layers:
# Extract weights if the layer contains parameters (skips MaxPool, Flatten, and Dropout)
    if len(layer.get_weights()) > 0:
        weights, biases = layer.get_weights()
        print(f"Extracting Layer: '{layer.name}' | Weights Shape: {weights.shape} | Biases Shape: {biases.shape}")

    # Save weights as hex values for SystemVerilog $readmemh
        weight_file_path = os.path.join(output_dir, f"{layer.name}_weights.hex")
        with open(weight_file_path, 'w') as f_weight:
            for w in weights.flatten():
                f_weight.write(f"{float_to_hex16(w, SCALE_FACTOR)}\n")

    # 2. Save biases as hex values for SystemVerilog $readmemh
        bias_file_path = os.path.join(output_dir, f"{layer.name}_biases.hex")
        with open(bias_file_path, 'w') as f_bias:
            for b in biases.flatten():
                f_bias.write(f"{float_to_hex16(b, SCALE_FACTOR)}\n")

print(f"Success: All parameters quantized and saved as hexadecimal files in the '/{output_dir}' directory.")


In [ ]:
# train_hybrid.py
# CNN + transformer
# achieved 95-96% accuracy



import setuptools
import sys

# Maintain environment stability
sys.modules['distutils'] = setuptools._distutils
import os

# ENVIRONMENT SETTINGS
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.decomposition import FastICA
import pywt  # Discrete Wavelet Transforms

print("--- Starting Hybrid CNN + Transformer ---", flush=True)

# DATA LOADING & MORPHOLOGICAL DSP PROCESSING

try:
    X_raw = np.load('X_final.npy')
    y = np.load('y_final.npy')
    if len(X_raw.shape) == 3:
        X_raw = X_raw.squeeze(-1)
    print(f"Loaded {X_raw.shape[0]} samples from data repository.", flush=True)
except FileNotFoundError:
    print("ERROR: Finalized data arrays not found.")
    sys.exit()

print("Executing 4-Level Daubechies-8 (db8) Wavelet Decomposition...", flush=True)
dwt_features = []
for sample in X_raw:
    # Extract structural detail coefficients to pinpoint QRS anomalies
    coeffs = pywt.wavedec(sample, 'db8', level=4)
    flattened_coeffs = np.concatenate([coeffs[0], coeffs[1], coeffs[2]])
    dwt_features.append(flattened_coeffs)
X_dwt = np.array(dwt_features)

print("Isolating Blind Sources via FastICA...", flush=True)
ica = FastICA(n_components=18, random_state=42, max_iter=1000)
X_ica = ica.fit_transform(X_raw)

# Combine into a single morphological array
X_morphology = np.hstack((X_dwt, X_ica))

print("Compressing Dimensionality via PCA (Targeting 26 Principal Components)...", flush=True)
pca = PCA(n_components=26, random_state=42)
X_pca_features = pca.fit_transform(X_morphology)

# Format raw input for the 1D-CNN temporal sequence branch
X_signal = X_raw.reshape(X_raw.shape[0], 180, 1)

# Parallel Train/Test Splitting
X_sig_train, X_sig_test, X_pca_train, X_pca_test, y_train, y_test = train_test_split(
    X_signal, X_pca_features, y, test_size=0.2, random_state=42
)



# DUAL-INPUT HYBRID CNN + TRANSFORMER ARCHITECTURE

def build_hybrid_network():
    #  BRANCH A: Temporal 1D-CNN + Self-Attention Pipeline
    sig_input = layers.Input(shape=(180, 1), name="Raw_ECG_Signal_Input")
    cnn = layers.Conv1D(32, kernel_size=5, activation='relu', padding='same')(sig_input)
    cnn = layers.MaxPooling1D(pool_size=2)(cnn)
    cnn = layers.Conv1D(64, kernel_size=3, activation='relu', padding='same')(cnn)

    # Project spatial maps into Transformer internal dimensions
    d_model = 64
    proj = layers.Dense(d_model)(cnn)

    # Tiny Transformer Multi-Head Self-Attention Layer
    attn = layers.MultiHeadAttention(num_heads=2, key_dim=d_model)(proj, proj)
    attn_normalized = layers.LayerNormalization()(attn + proj)
    flat_signal = layers.GlobalAveragePooling1D()(attn_normalized)

    #  BRANCH B: Hand-Crafted PCA Morphology Vector
    pca_input = layers.Input(shape=(26,), name="Engineered_PCA_Features_Input")
    dense_pca = layers.Dense(32, activation='relu')(pca_input)

    #  MULTI-MODAL FEATURE FUSION
    fused = layers.concatenate([flat_signal, dense_pca])
    shared = layers.Dense(32, activation='relu')(fused)
    dropout = layers.Dropout(0.3)(shared)
    output = layers.Dense(1, activation='sigmoid', name="Output_Classifier")(dropout)

    return models.Model(inputs=[sig_input, pca_input], outputs=output)


model = build_hybrid_network()
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.summary()


# TRAINING PIPELINE

print("\nTraining...", flush=True)
model.fit(
    x={"Raw_ECG_Signal_Input": X_sig_train, "Engineered_PCA_Features_Input": X_pca_train},
    y=y_train,
    validation_data=({"Raw_ECG_Signal_Input": X_sig_test, "Engineered_PCA_Features_Input": X_pca_test}, y_test),
    epochs=15,
    batch_size=32,
    verbose=1
)

# Save high-level functional model architecture
model.save('hybrid_ecg_model.h5')


# HARDWARE PARAMETER EXTRACTION

print("\n--- Initiating Automated Hardware Parameter Extraction ---")

SCALE_FACTOR = 4096  # 2^12 for Q4.12 fixed-point representation


def float_to_hex16(value, scale):
    int_val = int(round(value * scale))
    int_val = max(-32768, min(32767, int_val))  # Enforce signed 16-bit register boundaries
    if int_val < 0:
        int_val = (1 << 16) + int_val  # Generate clean 16-bit Two's Complement
    return f"{int_val:04X}"


output_dir = 'hardware_hybrid_roms'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Parse functional graph layers and unroll parameters for SystemVerilog compilation
for layer in model.layers:
    all_weights = layer.get_weights()
    if len(all_weights) > 0:
        print(f"Serializing Layer: '{layer.name}' | Contains {len(all_weights)} sub-tensors")

        # Open a unified hexadecimal memory file for the entire layer
        with open(os.path.join(output_dir, f"{layer.name}_parameters.hex"), 'w') as f_out:
            # Flatten every sub-tensor sequentially (Weights, Biases, Attention Kernels, etc.)
            for tensor_idx, tensor in enumerate(all_weights):
                print(f"  -> Processing Tensor index {tensor_idx} with shape {tensor.shape}...")
                for val in tensor.flatten():
                    f_out.write(f"{float_to_hex16(val, SCALE_FACTOR)}\n")

print(f"\nSuccess: All quantized hexadecimal parameters compiled cleanly in '/{output_dir}'.")

--- Starting Hybrid CNN + Transformer ---
Loaded 14258 samples from data repository.
Executing 4-Level Daubechies-8 (db8) Wavelet Decomposition...


/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 4 is too high: all coefficients will experience boundary effects.
  warnings.warn(


Isolating Blind Sources via FastICA...
Compressing Dimensionality via PCA (Targeting 26 Principal Components)...


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Raw_ECG_Signal_Inp… │ (None, 180, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 180, 32)   │        192 │ Raw_ECG_Signal_I… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_3     │ (None, 90, 32)    │          0 │ conv1d_6[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_7 (Conv1D)   │ (None, 90, 64)    │      6,208 │ max_pooling1d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 90, 64)    │      4,160 │ conv1d_7[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 90, 64)    │     33,216 │ dense_9[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 90, 64)    │          0 │ multi_head_atten… │
│                     │                   │            │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 90, 64)    │        128 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Engineered_PCA_Fea… │ (None, 26)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 32)        │        864 │ Engineered_PCA_F… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 96)        │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 32)        │      3,104 │ concatenate_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 32)        │          0 │ dense_11[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output_Classifier   │ (None, 1)         │         33 │ dropout_7[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 47,905 (187.13 KB)

 Trainable params: 47,905 (187.13 KB)

 Non-trainable params: 0 (0.00 B)


Training...
Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 30s 75ms/step - accuracy: 0.8115 - loss: 0.4124 - val_accuracy: 0.9278 - val_loss: 0.2444
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 27s 75ms/step - accuracy: 0.9240 - loss: 0.2369 - val_accuracy: 0.9478 - val_loss: 0.1724
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 41s 75ms/step - accuracy: 0.9408 - loss: 0.1856 - val_accuracy: 0.9471 - val_loss: 0.1602
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 26s 74ms/step - accuracy: 0.9469 - loss: 0.1627 - val_accuracy: 0.9558 - val_loss: 0.1363
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 26s 73ms/step - accuracy: 0.9497 - loss: 0.1481 - val_accuracy: 0.9537 - val_loss: 0.1202
Epoch 6/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 26s 74ms/step - accuracy: 0.9529 - loss: 0.1441 - val_accuracy: 0.9558 - val_loss: 0.1133
Epoch 7/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 26s 73ms/step - accuracy: 0.9554 - loss: 0.1287 - val_accuracy: 0.9562 - val_loss: 0.1100
Epoch 8/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 26s 74ms/step - accuracy: 0.9566 - lo


--- Initiating Automated Hardware Parameter Extraction ---
Serializing Layer: 'conv1d_6' | Contains 2 sub-tensors
  -> Processing Tensor index 0 with shape (5, 1, 32)...
  -> Processing Tensor index 1 with shape (32,)...
Serializing Layer: 'conv1d_7' | Contains 2 sub-tensors
  -> Processing Tensor index 0 with shape (3, 32, 64)...
  -> Processing Tensor index 1 with shape (64,)...
Serializing Layer: 'dense_9' | Contains 2 sub-tensors
  -> Processing Tensor index 0 with shape (64, 64)...
  -> Processing Tensor index 1 with shape (64,)...
Serializing Layer: 'multi_head_attention_3' | Contains 8 sub-tensors
  -> Processing Tensor index 0 with shape (64, 2, 64)...
  -> Processing Tensor index 1 with shape (2, 64)...
  -> Processing Tensor index 2 with shape (64, 2, 64)...
  -> Processing Tensor index 3 with shape (2, 64)...
  -> Processing Tensor index 4 with shape (64, 2, 64)...
  -> Processing Tensor index 5 with shape (2, 64)...
  -> Processing Tensor index 6 with shape (2, 64, 64)...


In [3]:
# train_hybrid_v2.py
# CNN + transformer
# achieved 98%+ accuracy


import os
import numpy as np
import scipy.signal as signal
import pywt  # To handle the Daubechies-8 Wavelet Transform
from sklearn.model_selection import train_test_split
from sklearn.decomposition import FastICA, PCA

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

# --- Environment Setup ---
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
print(" Starting Training (V2) ")

# DATA INGESTION

print("Loading clinical samples from data repository...")

X_raw_signal = np.load('X_final.npy').astype(np.float32)
y = np.load('y_final.npy').astype(np.float32)

print(f"Loaded raw signals with shape: {X_raw_signal.shape}")

print("Executing Zero-Phase Bidirectional Butterworth Filter (0.5Hz - 45Hz)...")


def apply_bidirectional_filter(data, fs=360.0):
    nyquist = 0.5 * fs
    low = 0.5 / nyquist
    high = 45.0 / nyquist
    b, a = signal.butter(3, [low, high], btype='band')

    filtered_data = np.zeros_like(data)
    for i in range(data.shape[0]):
        filtered_data[i, :] = signal.filtfilt(b, a, data[i, :])
    return filtered_data


# Clear out baseline wander noise
X_signal_clean = apply_bidirectional_filter(X_raw_signal)

print("Parallel DSP: Computing 4-Level DWT (db8) across windows...")
# Extract wavelet coefficients to capture multi-resolution QRS frequency spikes
dwt_features = []
for i in range(X_signal_clean.shape[0]):
    coeffs = pywt.wavedec(X_signal_clean[i, :], 'db8', level=4)
    flat_coeffs = np.concatenate(coeffs)
    dwt_features.append(flat_coeffs)
X_dwt = np.array(dwt_features)

print("Compressing Wavelet coefficients to 26 PCA Components...")
# Compress the high-dimensional DWT vector pool down to 26 structural core features
pca = PCA(n_components=26, random_state=42)
X_pca_compressed = pca.fit_transform(X_dwt)

print("Extracting 18-Basis FastICA source profiles...")
# Extract explicit physiological independent source pathways from raw data
ica = FastICA(n_components=18, random_state=42, max_iter=500)
X_ica_features = ica.fit_transform(X_signal_clean)

print("Merging PCA and ICA...")
# Concatenate parallel features into a single matrix
X_morph_features = np.concatenate([X_pca_compressed, X_ica_features], axis=-1).astype(np.float32)
print(f"-> Successfully generated hand-crafted feature matrix: {X_morph_features.shape}")

# Prepare raw signal inputs for the 1D-CNN
X_signal = np.expand_dims(X_signal_clean, axis=-1)

# Parallel Train/Test Splitting (80/20 split)
X_sig_train, X_sig_test, X_morph_train, X_morph_test, y_train, y_test = train_test_split(
    X_signal, X_morph_features, y, test_size=0.2, random_state=42
)




print("\nBuilding Functional Architecture...")

# Raw Temporal Signal Processing
raw_signal_input = layers.Input(shape=(180, 1), name="Raw_ECG_Signal_Input")
x = layers.Conv1D(32, kernel_size=5, padding='same', activation='relu', name="conv1d_v2_1")(raw_signal_input)
x = layers.MaxPooling1D(pool_size=2, name="max_pooling1d_v2_1")(x)
x = layers.SpatialDropout1D(0.2, name="spatial_dropout_1")(x)

x = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu', name="conv1d_v2_2")(x)
x = layers.SpatialDropout1D(0.2, name="spatial_dropout_2")(x)

# Project to Transformer internal space
x = layers.Dense(64, activation='relu', name="dense_projection")(x)

# Multi-Head Self-Attention Block
attention_out = layers.MultiHeadAttention(num_heads=2, key_dim=64, name="multi_head_attention_v2")(x, x)
x = layers.Add(name="residual_connection")([x, attention_out])
x = layers.LayerNormalization(name="layer_normalization_v2")(x)
flat_signal = layers.GlobalAveragePooling1D(name="global_average_pooling_v2")(x)

# Shape updated automatically to accept the total fused feature dimension (26 + 18 = 44)
morph_input = layers.Input(shape=(X_morph_features.shape[1],), name="Engineered_Morphological_Input")
y_dense = layers.Dense(64, activation='relu', name="morph_dense_1")(morph_input)
y_dense = layers.BatchNormalization(name="morph_batch_norm_1")(y_dense)
y_dense = layers.Dense(64, activation='relu', name="morph_dense_2")(y_dense)
y_dense = layers.Dense(32, activation='relu', name="morph_dense_3")(y_dense)

# --- Fusion & classification ---
fused_features = layers.Concatenate(name="symmetrical_fusion")([flat_signal, y_dense])
z = layers.Dense(32, activation='relu', name="dense_post_fusion")(fused_features)
z = layers.Dropout(0.3, name="classifier_dropout")(z)
output_classifier = layers.Dense(1, activation='sigmoid', name="Output_Classifier")(z)

model = models.Model(inputs=[raw_signal_input, morph_input], outputs=output_classifier)


# OPTIMIZATION & TRAINING LOOP

print("\nCommencing Training Loop...")

lr_decay = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
)

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    [X_sig_train, X_morph_train], y_train,
    validation_data=([X_sig_test, X_morph_test], y_test),
    epochs=15,
    batch_size=32,
    callbacks=[lr_decay]
)


# HARDWARE PARAMETER EXTRACTION

print("\n--- Initiating Hardware Parameter Extraction ---")

SCALE_FACTOR = 4096


def float_to_hex16(value, scale):
    int_val = int(round(value * scale))
    int_val = max(-32768, min(32767, int_val))
    if int_val < 0:
        int_val = (1 << 16) + int_val
    return f"{int_val:04X}"


output_dir = 'hardware_hybrid_v2_roms'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

for layer in model.layers:
    all_weights = layer.get_weights()
    if len(all_weights) > 0:
        print(f"Serializing Layer: '{layer.name}' | Sub-Tensors found: {len(all_weights)}")
        with open(os.path.join(output_dir, f"{layer.name}_parameters.hex"), 'w') as f_out:
            for tensor_idx, tensor in enumerate(all_weights):
                for val in tensor.flatten():
                    f_out.write(f"{float_to_hex16(val, SCALE_FACTOR)}\n")

print(f"\nSuccess: All quantized parameters compiled cleanly in '/{output_dir}'.")

 Starting Training (V2) 
Loading clinical samples from data repository...
Loaded raw signals with shape: (14258, 180)
Executing Zero-Phase Bidirectional Butterworth Filter (0.5Hz - 45Hz)...
Parallel DSP: Computing 4-Level DWT (db8) across windows...


/usr/local/lib/python3.12/dist-packages/pywt/_multilevel.py:43: UserWarning: Level value of 4 is too high: all coefficients will experience boundary effects.
  warnings.warn(


Compressing Wavelet coefficients to 26 PCA Components...
Extracting 18-Basis FastICA source profiles...
Merging PCA and ICA...
-> Successfully generated hand-crafted feature matrix: (14258, 44)

Building Functional Architecture...

Commencing Training Loop...
Epoch 1/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 20s 27ms/step - accuracy: 0.9500 - loss: 0.1356 - val_accuracy: 0.9849 - val_loss: 0.0466 - learning_rate: 0.0010
Epoch 2/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9824 - loss: 0.0505 - val_accuracy: 0.9891 - val_loss: 0.0357 - learning_rate: 0.0010
Epoch 3/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9869 - loss: 0.0367 - val_accuracy: 0.9884 - val_loss: 0.0278 - learning_rate: 0.0010
Epoch 4/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9902 - loss: 0.0311 - val_accuracy: 0.9916 - val_loss: 0.0281 - learning_rate: 0.0010
Epoch 5/15
357/357 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9916 - loss: 0.0241 - val_accuracy: 0.9909 - val_loss: 0.0304 - 